# 02 · 线性回归与逻辑回归 Linear Regression & Logistic Regression

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 区分**线性回归**（linear regression）与**逻辑回归**（logistic regression）的适用场景。
2. 推导 **sigmoid 函数**并理解其将线性输出映射为概率的机制。
3. 掌握 **L1（Lasso）/ L2（Ridge）正则化**对系数的影响。
4. 使用 `sklearn.linear_model` 拟合模型并输出预测概率。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 线性回归 Linear Regression

目标：拟合连续输出 $y \in \mathbb{R}$。

$$\hat{y} = \mathbf{w}^\top \mathbf{x} + b$$

最小化**均方误差**（MSE）：$\mathcal{L} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$

---

### Sigmoid 函数 & 逻辑回归

逻辑回归将线性打分 $z = \mathbf{w}^\top \mathbf{x} + b$ 转化为概率：

$$\sigma(z) = \frac{1}{1+e^{-z}} \in (0, 1)$$

| $z$ | $\sigma(z)$ |
|-----|------------|
| $-\infty$ | 0 |
| 0 | 0.5 |
| $+\infty$ | 1 |

损失函数：**交叉熵**（binary cross-entropy）

$$\mathcal{L} = -\frac{1}{n}\sum_i [y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$$

---

### 正则化 Regularization

| 类型 | 损失附加项 | 效果 |
|------|-----------|------|
| **L2 / Ridge** | $\lambda\|\mathbf{w}\|_2^2$ | 系数收缩，不为零 |
| **L1 / Lasso** | $\lambda\|\mathbf{w}\|_1$ | 系数稀疏（部分恰好=0）|

`LogisticRegression(penalty='l1', C=1.0)` 中 $C = 1/\lambda$（越小正则越强）。

## 示例 Worked Example — `fit_predict_proba`

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ── fit_predict_proba（镜像 w5-logreg 题包）───────────────────────────────
def fit_predict_proba(Xtr, ytr, Xte):
    """拟合逻辑回归，返回 Xte 的正类预测概率（一维数组）。"""
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1]

# ── 数据集 ────────────────────────────────────────────────────────────────
X, y = make_classification(n_samples=400, n_features=10, n_informative=5,
                            random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)

# ── 使用 fit_predict_proba ────────────────────────────────────────────────
proba = fit_predict_proba(Xtr_s, ytr, Xte_s)
auc = roc_auc_score(yte, proba)
print(f"逻辑回归 ROC-AUC = {auc:.4f}")

# ── Sigmoid 可视化 ─────────────────────────────────────────────────────────
z = np.linspace(-8, 8, 200)
sig = 1 / (1 + np.exp(-z))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Sigmoid curve
axes[0].plot(z, sig, "steelblue", lw=2)
axes[0].axhline(0.5, color="gray", ls="--", lw=1)
axes[0].axvline(0,   color="gray", ls="--", lw=1)
axes[0].set_title("Sigmoid 函数")
axes[0].set_xlabel("z")
axes[0].set_ylabel("σ(z)")
axes[0].set_ylim(-0.05, 1.05)

# L1/L2 正则化对系数的影响
Cs = np.logspace(-3, 2, 30)
coef_l1 = []
coef_l2 = []
for C in Cs:
    m1 = LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=1000)
    m2 = LogisticRegression(penalty="l2", C=C, solver="lbfgs",    max_iter=1000)
    m1.fit(Xtr_s, ytr); coef_l1.append(np.abs(m1.coef_[0]).mean())
    m2.fit(Xtr_s, ytr); coef_l2.append(np.abs(m2.coef_[0]).mean())

axes[1].plot(Cs, coef_l1, label="L1 (Lasso)", color="tomato")
axes[1].plot(Cs, coef_l2, label="L2 (Ridge)", color="steelblue")
axes[1].set_xscale("log")
axes[1].set_xlabel("C（正则强度的倒数）")
axes[1].set_ylabel("系数平均绝对值")
axes[1].set_title("L1 vs L2 正则化路径")
axes[1].legend()

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "logistic_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)

## 动手 Exercise

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# 练习：比较 L1、L2 正则化在不同 C 值下的 AUC
X, y = make_classification(n_samples=500, n_features=20, n_informative=5,
                            n_redundant=5, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)

print(f"{'penalty':>7}  {'C':>8}  {'AUC':>8}  {'非零系数':>8}")
print("-" * 40)
for penalty, solver in [("l1", "liblinear"), ("l2", "lbfgs")]:
    for C in [0.01, 0.1, 1.0, 10.0]:
        clf = LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=1000)
        clf.fit(Xtr_s, ytr)
        auc = roc_auc_score(yte, clf.predict_proba(Xte_s)[:, 1])
        nonzero = np.sum(clf.coef_[0] != 0)
        print(f"{penalty:>7}  {C:>8.2f}  {auc:>8.4f}  {nonzero:>8d}")
print("\n观察：L1 在小 C（强正则）时系数更稀疏；L2 系数收缩但不为零。")

## 延伸阅读 Further Reading

1. **sklearn LogisticRegression**：<https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression>
2. **《Pattern Recognition and Machine Learning》** Bishop — 第 4 章（线性判别模型）
3. **StatQuest — Logistic Regression**：<https://www.youtube.com/watch?v=yIYKR4sgzI8>
4. **正则化直观解释**：<https://towardsdatascience.com/l1-and-l2-regularization-methods-ce25e7fc831c>

## 关联练习 Related Assignments

```bash
python check.py w5-logreg
```

> 实现 `fit_predict_proba(Xtr, ytr, Xte)` — 拟合逻辑回归，返回正类概率（1-D 数组）。

> 能力标签：**SH7** · threshold ≥ 0.7